## Imports

In [ ]:
from pathlib import Path

import tiktoken
import torch

from nano_llm.attention import CausalAttention
from nano_llm.loaders import create_dataloader_v1

## Variables

In [ ]:
TEXT_PATH = Path("..") / "the-verdict.txt"
VOCAB_SIZE = tiktoken.get_encoding("gpt2").n_vocab
CONTEXT_LENGTH = 4
EMBEDDING_DIM = 256
DROPOUT_RATE = 0.1

raw_text = TEXT_PATH.read_text(encoding="utf-8")

## Loader

In [ ]:
dataloader = create_dataloader_v1(
    raw_text, batch_size=1, max_length=2, stride=2, shuffle=False
)

## Token embeddings

In [ ]:
torch.manual_seed(123)

dataloader = create_dataloader_v1(
    raw_text,
    batch_size=8,
    max_length=CONTEXT_LENGTH,
    stride=CONTEXT_LENGTH,
    shuffle=False,
)
inputs, targets = next(iter(dataloader))

token_embedding_layer = torch.nn.Embedding(VOCAB_SIZE, EMBEDDING_DIM)
token_embeddings = token_embedding_layer(inputs)

print("Token IDs:\n", inputs)
print("Token embeddings shape:", token_embeddings.shape)

## Positional embeddings

In [ ]:
pos_embedding_layer = torch.nn.Embedding(CONTEXT_LENGTH, EMBEDDING_DIM)
pos_embeddings = pos_embedding_layer(torch.arange(CONTEXT_LENGTH))

input_embeddings = token_embeddings + pos_embeddings

print("Positional embeddings shape:", pos_embeddings.shape)
print("Input embeddings shape:", input_embeddings.shape)

## Self-Attention

In [ ]:
torch.manual_seed(123)

self_attention_layer = CausalAttention(
    EMBEDDING_DIM, EMBEDDING_DIM, CONTEXT_LENGTH, DROPOUT_RATE
)
context_vectors = self_attention_layer(input_embeddings)

print("Input embeddings shape:", input_embeddings.shape)
print("Context vectors shape:", context_vectors.shape)